# Example Validation Workflow

**Phase 1B Infrastructure - Complete Example**

This notebook demonstrates how to use the validation infrastructure built in Phase 1B.1.

**Contents**:
1. Setup and imports
2. Generate synthetic data
3. Run validation
4. Visualize results
5. Parallel simulations
6. Storage and retrieval

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

from dml_ts.validation import (
    DGPGenerator, DGPResult, ResultStorage, ValidationResult, parallel_monte_carlo, plotting,
)

## 2. Generate Synthetic Data

Create a data generating process (DGP) with known treatment effect.

In [ ]:
dgp_params = dict(n=1000, p=5, true_effect=2.0, confounding_strength=1.0,
                  treatment_model="linear", outcome_model="linear")
dgp = DGPGenerator(**dgp_params, random_state=42)

data = dgp.generate()

print(f"Generated data:")
print(f"  Y (outcome): shape {data.Y.shape}")
print(f"  T (treatment): shape {data.T.shape}")
print(f"  X (confounders): shape {data.X.shape}")
print(f"  True effect (τ): {data.true_effect}")

### Visualize Generated Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Treatment distribution
axes[0].hist(data.T, bins=30, edgecolor="white")
axes[0].set_xlabel("Treatment (T)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Treatment Distribution")

# Outcome distribution
axes[1].hist(data.Y, bins=30, edgecolor="white", color="orange")
axes[1].set_xlabel("Outcome (Y)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Outcome Distribution")

# Treatment vs Outcome
axes[2].scatter(data.T, data.Y, alpha=0.5)
axes[2].set_xlabel("Treatment (T)")
axes[2].set_ylabel("Outcome (Y)")
axes[2].set_title("Treatment vs Outcome")

plt.tight_layout()
plt.show()

## 3. Simple estimator example

Compare the treatment-only slope to the slope adjusted for the five observed confounders. The linear outcome model is correctly specified after adjustment. The generator redraws the confounding coefficients for each dataset, so the sign of omitted-variable bias can vary across runs.

In [ ]:
def ols_fit(data: DGPResult, adjust: bool):
    design = np.column_stack([data.T, data.X]) if adjust else data.T
    return sm.OLS(data.Y, sm.add_constant(design)).fit()

print(f"True effect: {data.true_effect:.3f}")
for label, adjust in [("Treatment only", False), ("With controls", True)]:
    estimate = ols_fit(data, adjust).params[1]
    print(f"{label}: {estimate:.3f} (single-dataset error {estimate - data.true_effect:+.3f})")

## 4. Monte Carlo validation

Across 1,000 independent datasets, compare each adjusted slope and its own 95% OLS interval with the true effect. Estimate mean bias and its Monte Carlo uncertainty separately. The demonstration flags a warning when the 95% interval for mean bias lies outside the equivalence criterion in whole or in part (a tolerance of ±0.01 effect units), or the exact binomial test rejects 95% coverage at 0.05. This example uses independent observations and homoskedastic normal regression errors.

In [ ]:
def run_simulation(sim_id: int, dgp_params: dict):
    sample = DGPGenerator(**dgp_params, random_state=sim_id).generate()
    fit = ols_fit(sample, adjust=True)
    lower, upper = fit.conf_int(alpha=0.05)[1]
    return fit.params[1], lower, upper

n_simulations = 1000
first_seed = 20261121
results = np.asarray([run_simulation(first_seed + i, dgp_params) for i in range(n_simulations)])
estimates, lower, upper = results.T
biases = estimates - dgp.true_effect
mean_bias = biases.mean()
mcse = stats.sem(biases)
bias_ci = mean_bias + np.array([-1, 1]) * stats.t.ppf(.975, n_simulations - 1) * mcse
covered = (lower <= dgp.true_effect) & (dgp.true_effect <= upper)
coverage_p = stats.binomtest(int(covered.sum()), n_simulations, p=.95).pvalue

print(f"Mean estimate: {estimates.mean():.4f}, mean bias: {mean_bias:+.4f}, MC SE: {mcse:.4f}")
print(f"95% interval for mean bias: [{bias_ci[0]:+.4f}, {bias_ci[1]:+.4f}]")
print(f"RMSE: {np.sqrt(np.mean(biases**2)):.4f}, coverage: {covered.sum()}/{n_simulations}")

### Visualize Bias Distribution

In [ ]:
# Use built-in plotting utility
fig = plotting.plot_bias_distribution(
    biases=biases, title="Sampling distribution of adjusted OLS estimation errors"
)
plt.show()

## 5. Parallel Execution

Use the parallel execution framework for faster simulations.

In [ ]:
# Reuse identical seeds and data-generating parameters in both execution modes.
parallel_results = parallel_monte_carlo(
    run_simulation, n_simulations=n_simulations, n_jobs=4, show_progress=False,
    random_state=first_seed, dgp_params=dgp_params,
)
np.testing.assert_allclose(parallel_results, results, rtol=1e-12, atol=1e-12)
print(f"Serial and parallel estimates and interval bounds agree in all {n_simulations} draws.")

## 6. Storage and Retrieval

Save and load validation results.

In [ ]:
from datetime import datetime

bias_status = "PASS" if bias_ci[0] >= -.01 and bias_ci[1] <= .01 else "WARNING"
coverage_status = "PASS" if coverage_p >= .05 else "WARNING"
result = ValidationResult(
    method="OLS_with_controls",
    status="PASS" if bias_status == coverage_status == "PASS" else "WARNING",
    bias=mean_bias, mse=np.mean(biases**2), coverage=covered.mean(),
    ci_lower=bias_ci[0], ci_upper=bias_ci[1], n_simulations=n_simulations,
    timestamp=datetime.now(), metadata={"dgp": dgp_params, "bias_tolerance": .01},
    ci_estimates=results[:, 1:], point_estimates=estimates,
    bias_status=bias_status, coverage_status=coverage_status, coverage_p_value=coverage_p,
)
print(result.summary())

In [ ]:
# Save to storage
storage = ResultStorage(base_dir="../results/validation")
filepath = storage.save_result(result, metadata={"notebook": "example_validation"})

print(f"✅ Result saved to: {filepath}")

In [ ]:
# Load results
loaded_results = storage.load_results(limit=5)

print(f"Loaded {len(loaded_results)} results:")
for r in loaded_results:
    print(f"  - {r.method}: {r.status} (bias={r.bias:.3f}, RMSE={np.sqrt(r.mse):.3f})")

In [ ]:
# Get storage summary
summary = storage.get_summary()
print(f"\nStorage Summary:")
print(f"  Total results: {summary['total_results']}")
print(f"  Methods: {summary['methods']}")
print(f"  Statuses: {summary['statuses']}")
print(f"  Cache entries: {summary['cache_entries']}")

## 7. Next Steps

**You've now seen the complete validation workflow!**

To implement your own validation method:

1. Copy `templates/validation_method_template.py`
2. Implement your estimator in `_estimate_effect()`
3. Copy `templates/test_harness_template.py` and write tests
4. Run `pytest` to verify
5. Use `parallel_monte_carlo` for large simulations
6. Generate reports with `plotting` utilities

[Historical validation implementation guide](../docs/archive/superseded_2026-05-02/VALIDATION_IMPLEMENTATION_GUIDE.md)

---

**Generated**: 2025-11-13  
**Phase**: 1B.1 Infrastructure  
**Project**: Double ML Validation Framework  